# 实践项目 01：MRI 脑肿瘤图像分割

这份 Notebook 从配对的二维 MRI 切片和肿瘤 mask 开始，完成数据核对、患者级划分、同步预处理、轻量 U-Net、训练、阈值选择和测试评价。MRI 是模型看到的输入，mask 是训练和评价时表示肿瘤区域的目标。

Kaggle Notebook 是本项目的首选实践入口。打开公开 Notebook 后，点击“复制并编辑”保存到自己的账户，再按顺序运行单元格；下载到电脑运行是补充方式。每个任务都写明了输入、输出和需要填写的位置。

生成的图像和 JSON 只用于自己的观察和复盘。

**学生需要填写或修改的位置：** 带有 `TODO`、`None` 占位或“你的记录”的区域。先看 shape、输入输出说明和断言，再填写当前数据产生的结果，不要把参考数字写死。

## 任务总览

1. 查找 MRI 与 mask 并确认一一配对。
2. 统计患者数量、阳性 mask 数量和空 mask 比例。
3. 按患者划分训练集、验证集和测试集。
4. 把图像和 mask 同步缩放，训练集只增加水平翻转。
5. 补全 Dice 和 U-Net 卷积块。
6. 完成训练更新并保存训练曲线。
7. 只用验证集选择阈值，再在测试集展示预测结果。


## 需要保存的结果

- `task1_data_check.png`：一张 MRI、对应 mask 和叠加图。
- `task1_training_curve.png`：训练/验证损失和 Dice 曲线。
- `task1_prediction.png`：测试图像、真实 mask 和预测 mask。
- `task1_result.json`：数据规模、阈值、指标和空 mask 统计。

这些文件由 Notebook 自动写入 `OUT` 指向的工作目录，用于自己的观察和复盘。


In [ ]:
# 输入：无；输出：数据读取、训练和结果保存所需的库与路径。
from pathlib import Path
import json, random, re
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import GroupShuffleSplit

SEED=42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
DEVICE=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
OUT=Path('/kaggle/working') if Path('/kaggle/working').exists() else Path.cwd()/'kydw_outputs'
OUT.mkdir(parents=True,exist_ok=True)
print('device:',DEVICE)


## 1. 查找并配对图像与 mask

**输入：** 推荐挂载 `lgg-mri-segmentation` 数据集。MRI 文件与 mask 文件在文件名上只相差 `_mask`，患者标识取自 mask 的上级文件夹。

**处理：** 代码遍历数据目录，把每个 mask 与去掉 `_mask` 后仍然存在的 MRI 配成一项 `(image_path, mask_path, patient_id)`。

**输出：** `pairs` 列表和 `paired slices` 数量。数量应为正数；后面的断言用来检查数据路径和文件名规则是否正确。这一段没有待填写代码，先运行并观察输出。


In [ ]:
# 输入：数据集根目录；输出：每项 (image_path, mask_path, patient_id) 的 pairs。
ROOT=Path('/kaggle/input') if Path('/kaggle/input').exists() else Path.cwd()/'data'
all_tif=[p for p in ROOT.rglob('*') if p.is_file() and p.suffix.lower() in {'.tif','.tiff','.png'} and '__MACOSX' not in p.parts and not p.name.startswith('._')]
mask_paths=[p for p in all_tif if p.stem.lower().endswith('_mask')]
pairs=[]
for m in mask_paths:
    img=m.with_name(re.sub(r'_mask(?=\.[^.]+$)', '', m.name, flags=re.IGNORECASE))
    if img.exists():
        patient=m.parent.name
        pairs.append((img,m,patient))
print('paired slices:',len(pairs))
assert pairs, '没有找到配对 MRI 与 mask，请查看额外注意事项中的数据集挂载说明。'


## 任务 1：完成数据核对

这一步先确认输入数据真的可以使用。阳性 mask 表示其中至少有一个前景像素；空 mask 表示整张 mask 没有前景。

**学生需要填写：** 根据 `pairs` 中的患者标识和每个 mask 的实际像素，计算唯一患者数 `patient_count`、阳性 mask 数 `positive_masks` 和空 mask 比例 `empty_ratio`。

**输入与输出：** 输入是 `pairs`；输出是三个标量，以及一张 MRI、mask 和 overlay 图。单张 `img` 与 `mask` 都应是 `[H, W]`，尺寸必须一致。

**检查：** 统计只能来自当前文件；运行后应打印三个统计量，并保存 `task1_data_check.png`。


In [ ]:
# ===== 请在此处完成：项目01·任务1 患者与 mask 数据统计（开始） =====
# TODO 1
patient_count = None
positive_masks = None
empty_ratio = None
# ===== 请在此处完成：项目01·任务1 患者与 mask 数据统计（结束） =====

sample_img_path, sample_mask_path, _ = pairs[len(pairs)//2]
img=np.array(Image.open(sample_img_path).convert('L'))
mask=np.array(Image.open(sample_mask_path).convert('L'))>0
print(patient_count, positive_masks, empty_ratio, img.shape, mask.shape)

fig,ax=plt.subplots(1,3,figsize=(10,3))
ax[0].imshow(img,cmap='gray'); ax[0].set_title('MRI')
ax[1].imshow(mask,cmap='gray'); ax[1].set_title('mask')
ax[2].imshow(img,cmap='gray'); ax[2].imshow(mask,alpha=.4,cmap='viridis'); ax[2].set_title('overlay')
for a in ax:a.axis('off')
plt.tight_layout(); plt.savefig(OUT/'task1_data_check.png',dpi=160); plt.show()

## 2. 患者级划分

同一患者可能有多张切片。如果把同一患者的切片拆到不同集合，测试评价会偏乐观。因此这里按 `patient_id` 分组划分。

**输入：** `pairs` 中的图像、mask 和患者标识。

**输出：** `tr_idx`、`va_idx`、`test_idx` 三组切片索引，以及三个互不重叠的患者集合。代码底部的交集断言应通过。


In [ ]:
# 输入：pairs 中的患者分组；输出：患者不重叠的 tr_idx、va_idx、test_idx。
MAX_SLICES=min(1200,len(pairs))
pairs=pairs[:MAX_SLICES]
groups=np.array([x[2] for x in pairs]); idx=np.arange(len(pairs))
gss=GroupShuffleSplit(n_splits=1,test_size=.20,random_state=SEED)
train_idx,test_idx=next(gss.split(idx,groups=groups))
train_groups=groups[train_idx]
gss2=GroupShuffleSplit(n_splits=1,test_size=.20,random_state=SEED)
tr_rel,va_rel=next(gss2.split(train_idx,groups=train_groups))
tr_idx=train_idx[tr_rel]; va_idx=train_idx[va_rel]
assert not (set(groups[tr_idx]) & set(groups[va_idx]) | set(groups[tr_idx]) & set(groups[test_idx]) | set(groups[va_idx]) & set(groups[test_idx]))
print(len(tr_idx),len(va_idx),len(test_idx))


In [ ]:
# 输入：pairs 和三组切片索引；输出：训练、验证、测试 DataLoader。
class MRIDataset(Dataset):
    def __init__(self,pairs,indices,size=128,augment=False):
        self.items=[pairs[i] for i in indices]; self.size=size; self.augment=augment
    def __len__(self): return len(self.items)
    def __getitem__(self,i):
        ip,mp,pid=self.items[i]
        image=Image.open(ip).convert('L').resize((self.size,self.size))
        mask=Image.open(mp).convert('L').resize((self.size,self.size),Image.Resampling.NEAREST)
        x=np.asarray(image,dtype=np.float32)/255.0
        y=(np.asarray(mask)>0).astype(np.float32)
        if self.augment and random.random()<.5:
            x=np.fliplr(x).copy(); y=np.fliplr(y).copy()
        return torch.from_numpy(x[None]),torch.from_numpy(y[None]),pid

train_ds=MRIDataset(pairs,tr_idx,augment=True); val_ds=MRIDataset(pairs,va_idx); test_ds=MRIDataset(pairs,test_idx)
train_loader=DataLoader(train_ds,batch_size=16,shuffle=True,num_workers=2)
val_loader=DataLoader(val_ds,batch_size=16,shuffle=False,num_workers=2)
test_loader=DataLoader(test_ds,batch_size=16,shuffle=False,num_workers=2)


## 任务 2：补全 Dice

Dice 用预测前景与真实前景的重叠程度评价分割结果。`prob` 是模型经过 sigmoid 后的概率图，`target` 是 0/1 mask；概率大于等于 `threshold` 的像素属于预测前景。

**学生需要填写：** 在 `dice_score` 的标记区域中完成阈值化、交集、预测面积和真实面积的计算，并返回 batch 内样本 Dice 的平均值。

**输入与输出：** `prob`、`target` 和 `pred` 的 shape 都是 `[batch, 1, H, W]`；沿空间维度求和后，每个样本得到一个 Dice，函数最后输出一个标量。

**检查：** 使用给定的 `threshold`、`eps` 和空间维度；`soft_dice_score` 已提供给训练损失使用。后面的训练应能得到有限的 Dice。


In [ ]:
def dice_score(prob,target,threshold=.5,eps=1e-6):
# ===== 请在此处完成：项目01·任务2 阈值化 Dice（开始） =====
    # TODO 2：阈值化、计算交集与两侧面积
    return None
# ===== 请在此处完成：项目01·任务2 阈值化 Dice（结束） =====

def soft_dice_score(prob,target,eps=1e-6):
    dims=tuple(range(1,prob.ndim))
    inter=(prob*target).sum(dims)
    return (2*inter+eps)/(prob.sum(dims)+target.sum(dims)+eps)

## 任务 3：补全 U-Net 卷积块

`DoubleConv` 是 U-Net 中重复使用的小模块：两次 `3×3` 卷积，每次卷积后进行归一化和 ReLU。`padding=1` 让高度和宽度保持不变，便于后面与跳跃连接拼接。

**学生需要填写：** 用 `cin`、`cout` 构造 `self.block`，顺序为卷积、归一化、ReLU、卷积、归一化、ReLU。不要改动 `TinyUNet` 中已经给出的连接。

**输入与输出：** 输入是 `[batch, cin, H, W]`，输出是 `[batch, cout, H, W]`。各层的 `cin`、`cout` 已由 `TinyUNet` 给出。

**检查：** 运行模型构造代码并查看参数量；后续前向传播的通道维度应能顺利连接。


In [ ]:
class DoubleConv(nn.Module):
    def __init__(self,cin,cout):
        super().__init__()
# ===== 请在此处完成：项目01·任务3 DoubleConv 卷积块（开始） =====
        # TODO 3
        self.block = None
# ===== 请在此处完成：项目01·任务3 DoubleConv 卷积块（结束） =====
    def forward(self,x): return self.block(x)

class TinyUNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.e1=DoubleConv(1,16); self.p1=nn.MaxPool2d(2)
        self.e2=DoubleConv(16,32); self.p2=nn.MaxPool2d(2)
        self.b=DoubleConv(32,64)
        self.u2=nn.ConvTranspose2d(64,32,2,2); self.d2=DoubleConv(64,32)
        self.u1=nn.ConvTranspose2d(32,16,2,2); self.d1=DoubleConv(32,16)
        self.out=nn.Conv2d(16,1,1)
    def forward(self,x):
        e1=self.e1(x); e2=self.e2(self.p1(e1)); b=self.b(self.p2(e2))
        d2=self.d2(torch.cat([self.u2(b),e2],1))
        d1=self.d1(torch.cat([self.u1(d2),e1],1))
        return self.out(d1)

model=TinyUNet().to(DEVICE)
print('parameters:',sum(p.numel() for p in model.parameters()))

## 任务 4：补全一次训练更新

训练分支每批先清空旧梯度，再计算 logits 和损失，反向传播并更新参数。验证分支只计算损失和 Dice，不更新参数。

**学生需要填写：** 在第一个标记区域填写清梯度代码，在第二个标记区域填写反向传播和优化器更新代码。其他损失函数、20 轮训练和最佳验证 Dice 选择逻辑保持不变。

**输入与输出：** `x`、`y` 和 `logits` 都是 `[batch, 1, H, W]`；`loss` 和每批 Dice 是标量。输出是每轮训练/验证损失和 Dice，以及训练曲线文件。

**检查：** 20 轮训练应打印 history，最佳模型应能载入，训练曲线应保存成功。


In [ ]:
bce=nn.BCEWithLogitsLoss()
opt=torch.optim.Adam(model.parameters(),lr=1e-3)

def run_epoch(loader,training):
    model.train(training); losses=[]; dices=[]
    for x,y,_ in loader:
        x=x.to(DEVICE); y=y.to(DEVICE)
# ===== 请在此处完成：项目01·任务4 清空训练梯度（开始） =====
        if training:
            # TODO 4：清梯度
            pass
# ===== 请在此处完成：项目01·任务4 清空训练梯度（结束） =====
        logits=model(x); prob=torch.sigmoid(logits)
        loss=bce(logits,y)+(1-soft_dice_score(prob,y)).mean()
# ===== 请在此处完成：项目01·任务4 反向传播与更新（开始） =====
        if training:
            # TODO 4：反向传播与更新
            pass
# ===== 请在此处完成：项目01·任务4 反向传播与更新（结束） =====
        losses.append(float(loss.detach().cpu())); dices.append(float(dice_score(prob,y).detach().cpu()))
    return np.mean(losses),np.mean(dices)

history=[]; best=None; best_d=-1
for epoch in range(20):
    tl,td=run_epoch(train_loader,True); vl,vd=run_epoch(val_loader,False)
    history.append((tl,td,vl,vd)); print(epoch+1,history[-1])
    if vd>best_d: best_d=vd; best={k:v.detach().cpu().clone() for k,v in model.state_dict().items()}
model.load_state_dict(best)

In [ ]:
# 输入：history 中每轮的 loss 和 Dice；输出：task1_training_curve.png。
h=np.array(history)
fig,ax=plt.subplots(1,2,figsize=(9,3.5))
ax[0].plot(h[:,0],label='train'); ax[0].plot(h[:,2],label='validation'); ax[0].set_title('loss'); ax[0].legend()
ax[1].plot(h[:,1],label='train'); ax[1].plot(h[:,3],label='validation'); ax[1].set_title('Dice'); ax[1].legend()
plt.tight_layout(); plt.savefig(OUT/'task1_training_curve.png',dpi=160); plt.show()


## 任务 5：测试评价与阈值比较

模型输出概率图，需要用阈值变成二值预测 mask。先在验证集比较给定的候选阈值，再把验证集 Dice 最高的阈值固定下来，最后评价测试集。

**学生需要填写：** 在标记区域遍历验证集，计算三个候选阈值对应的 `val_dices`，并从给定列表中选出 `best_threshold`。测试集不能参与阈值选择。

**输入与输出：** 输入是最佳模型、`val_loader` 和候选阈值；输出是阈值到 Dice 的字典和一个候选浮点数。后面的完整 `evaluate` 函数会返回测试 Dice、IoU、预测样本和空 mask 统计。

**检查：** 所有候选阈值都应有数值，`best_threshold` 必须属于代码中给定的列表。


In [ ]:
# ===== 请在此处完成：项目01·任务5 验证集阈值选择（开始） =====
# 输入：model、val_loader 和候选 thresholds；输出：val_dices、best_threshold。
thresholds = [.3, .5, .7]
val_dices = {}

# 下面的 evaluate 已经完整提供；本任务只补全阈值遍历和最佳阈值选择。
def evaluate(loader, threshold):
    model.eval(); ds=[]; ious=[]; positive_ds=[]; positive_ious=[]; examples=[]
    empty_true=empty_pred=both_empty=0
    with torch.no_grad():
        for x,y,pid in loader:
            prob=torch.sigmoid(model(x.to(DEVICE))).cpu(); pred=(prob>=threshold).float()
            target_area=y.sum((1,2,3)); pred_area=pred.sum((1,2,3))
            empty_true += int((target_area==0).sum()); empty_pred += int((pred_area==0).sum())
            both_empty += int(((target_area==0)&(pred_area==0)).sum())
            inter=(pred*y).sum((1,2,3)); union=((pred+y)>0).float().sum((1,2,3))
            d=(2*inter+1e-6)/(pred.sum((1,2,3))+y.sum((1,2,3))+1e-6)
            i=(inter+1e-6)/(union+1e-6)
            positive=target_area>0
            ds.extend(d.numpy()); ious.extend(i.numpy()); positive_ds.extend(d[positive].numpy()); positive_ious.extend(i[positive].numpy())
            if len(examples)<4:
                for k in range(min(4-len(examples),len(x))): examples.append((x[k,0].numpy(),y[k,0].numpy(),prob[k,0].numpy()))
    stats={'total':len(ds),'empty_true':empty_true,'empty_pred':empty_pred,'both_empty':both_empty,'positive_true':len(positive_ds),'positive_dice':float(np.mean(positive_ds)) if positive_ds else None,'positive_iou':float(np.mean(positive_ious)) if positive_ious else None}
    return float(np.mean(ds)),float(np.mean(ious)),examples,stats
# TODO 5：遍历验证集，计算每个 threshold 的 Dice。
# TODO 5：从 val_dices 中选择验证 Dice 最高的候选阈值。
best_threshold = None
# ===== 请在此处完成：项目01·任务5 验证集阈值选择（结束） =====


## 测试集评价与结果保存

这里是任务 5 的后半段，不是新的任务。阈值确定后，下面的 `evaluate` 才读取测试集；它返回整体 Dice/IoU、可视化样本和空 mask 统计。

**输入：** 测试 DataLoader、最佳阈值和已经载入最佳验证模型。

**输出：** `task1_prediction.png` 和 `task1_result.json`。图像帮助观察边界偏差，JSON 保存本次运行的数量、阈值和指标。


In [ ]:
# evaluate 已在上一个代码单元格提供；这里使用选出的阈值评价测试集。

test_dice,test_iou,examples,test_stats=evaluate(test_loader,best_threshold)
print(best_threshold,test_dice,test_iou)


In [ ]:
# 输入：examples、best_threshold 和测试指标；输出：预测图和 task1_result.json。
fig,ax=plt.subplots(len(examples),3,figsize=(8,2.5*len(examples)))
for r,(x,y,p) in enumerate(examples):
    ax[r,0].imshow(x,cmap='gray'); ax[r,0].set_title('MRI')
    ax[r,1].imshow(y,cmap='gray'); ax[r,1].set_title('true mask')
    ax[r,2].imshow(x,cmap='gray'); ax[r,2].imshow(p>=best_threshold,alpha=.4,cmap='viridis'); ax[r,2].set_title('prediction')
    for c in range(3): ax[r,c].axis('off')
plt.tight_layout(); plt.savefig(OUT/'task1_prediction.png',dpi=160); plt.show()

result={'train_slices':len(train_ds),'validation_slices':len(val_ds),'test_slices':len(test_ds),'best_threshold':best_threshold,'test_dice':test_dice,'test_iou':test_iou,'test_mask_stats':test_stats,'seed':SEED}
(OUT/'task1_result.json').write_text(json.dumps(result,indent=2),encoding='utf-8')
result


## 结果说明

结合数据配对、患者级划分、最佳阈值、测试 Dice/IoU、空 mask 统计和一个预测偏差，说明图像结果与数字指标如何相互支持。

**学生需要记录：** 数据配对与患者级划分、最佳阈值及依据、测试 Dice/IoU、空 mask 统计、一个预测偏差及其图像证据。记录只用于自己的理解。

<!-- ===== 请在此处完成：项目01·结果说明记录（开始） ===== -->
**你的记录：**

- 数据配对与患者级划分：
- 最佳阈值及依据：
- 测试 Dice/IoU：
- 空 mask 统计：
- 预测偏差与图像证据：
<!-- ===== 请在此处完成：项目01·结果说明记录（结束） ===== -->
